# Configuração do Ambiente

Criação do catálogo e schema da camada Silver.

In [0]:
spark.sql("CREATE catalog IF NOT EXISTS cinedata_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS cinedata_lakehouse.silver")
spark.sql("USE CATALOG cinedata_lakehouse")
spark.sql("USE SCHEMA silver")

# Processamento: tb_info_filmes

Limpeza, tradução de status, deduplicação e tratamento formato de datas dos metadados dos filmes.

In [0]:
from pyspark.sql.functions import trim,lower,regexp_replace,when,col
  
df_movies_info = spark.read.table("bronze.tb_movies_info")

#normalização da coluna status
df_status_normalizado = df_movies_info.withColumn(
    "status_limpo",
    trim(lower(regexp_replace(col("status"),"[-_]"," ")))
    ).drop("status")
#tradução da coluna status
df_status_traduzido = df_status_normalizado.withColumn(
    "status_filme",
    when (col("status_limpo") == "released","Lançado")
    .when(col("status_limpo") == "post production","Pós-Produção")
    .when(col("status_limpo") == "planned","Planejado")
    .when(col("status_limpo") == "in production","Em Produção")
    .when(col("status_limpo") == "rumored","Rumores")
    .when(col("status_limpo") == "canceled","Cancelado")   
    .otherwise("Não Informado")
).drop("status_limpo")

In [0]:
from pyspark.sql.window import Window 
from pyspark.sql.functions import col, row_number

#deduplicação pelo id
janela_filme = Window.partitionBy("id").orderBy(col("ingestion_datetime").desc())
df_deduplicado = (
    df_status_traduzido
    .withColumn("row_num", row_number().over(janela_filme))
    .filter(col("row_num") == 1)
    .drop("row_num")
)


In [0]:
from pyspark.sql.functions import col, coalesce, try_to_date, year
#tratamento da data em seus possiveis formatos
df_info_datas = df_deduplicado.withColumn(
    "data_lancamento",
    coalesce(
        try_to_date(col("release_date"),"yyyy-MM-dd"),
        try_to_date(col("release_date"),"dd-MM-yyyy"),
        try_to_date(col("release_date"),"MM-dd-yyyy"),
        try_to_date(col("release_date"),"yyyy/MM/dd"),
        try_to_date(col("release_date"),"dd/MM/yyyy"),
        try_to_date(col("release_date"),"MM/dd/yyyy")
)
).withColumn(
    "ano_lancamento",
     year(col("data_lancamento"))
    ).drop("release_date")

In [0]:
df_silver_info_filmes = (
    df_info_datas
    .withColumnRenamed("id","id_filme")
    #mantendo a coluna tconst renomeada para id_imdb por ser um identificador tradicional
    .withColumnRenamed("tconst","id_imdb")
    .withColumnRenamed("title","titulo")
    .withColumnRenamed("original_title","titulo_original")
    .withColumnRenamed("original_language","idioma_original")
    .withColumnRenamed("runtime","duracao_minutos")
    .withColumnRenamed("overview","sinopse")
    .withColumnRenamed("tagline","frase_divulgacao")
)

In [0]:
from pyspark.sql.functions import col, when, length, expr

# Limpeza estrutural de todas as colunas vulneráveis da tabela de informações
df_silver_info_treated = df_silver_info_filmes.withColumn(
    "duracao_minutos",
    # Tenta converter para inteiro; se for texto deslocado ou número negativo, vira NULL
    when(expr("try_cast(duracao_minutos as int)") >= 0, expr("try_cast(duracao_minutos as int)"))
    .otherwise(None)
).withColumn(
    "idioma_original",
    # Protege a coluna de idioma contra textos longos
    when(length(col("idioma_original")) <= 2, col("idioma_original"))
    .otherwise("Não Informado")
).withColumn(
    "status_filme",
    # Garante que apenas os status mapeados sobrevivam
    when(col("status_filme").isin("Lançado", "Pós-Produção", "Em Produção", "Planejado", "Rumores", "Cancelado"), col("status_filme"))
    .otherwise("Não Informado")
)

In [0]:
(
    df_silver_info_treated.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_info_filmes")
)

# Processamento: tb_financeiro_filmes

Deduplicação, limpeza textual, conversão decimal segura e cruzamento com info filmes e cotação do dólar.

In [0]:
from pyspark.sql.functions import col, row_number, when, regexp_replace, expr
from pyspark.sql.window import Window

# Ler a Bronze e DEDUPLICAR pelo id
df_bronze_fin = spark.read.table("bronze.tb_movies_financials")
janela_fin = Window.partitionBy("id").orderBy(col("ingestion_datetime").desc())

df_fin_dedup = df_bronze_fin.withColumn(
    "row_num", row_number().over(janela_fin)
).filter(col("row_num") == 1).drop("row_num")

In [0]:
# Tradução de Colunas
df_fin_traduzido = ( 
    df_fin_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("tconst", "id_imdb")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
)

In [0]:
# Limpeza Textual - Financeiro
df_fin_limpo = df_fin_traduzido.withColumn(
    "orcamento_usd",
    when(col("orcamento_usd").isin("Unknown","Não Informado"), None)
    .otherwise(regexp_replace(col("orcamento_usd"), "[^0-9.]", ""))
).withColumn(
    "receita_usd",
    when(col("receita_usd").isin("Unknown","Não Informado"), None)
    .otherwise(regexp_replace(col("receita_usd"), "[^0-9.]", ""))
)

In [0]:
#Conversão Decimal Segura
df_fin_decimal = df_fin_limpo.withColumn(
    "orcamento_usd",
    when(expr("try_cast(orcamento_usd as decimal(18,2))") <= 0, None)
    .otherwise(expr("try_cast(orcamento_usd as decimal(18,2))"))
).withColumn(
    "receita_usd",
    when(expr("try_cast(receita_usd as decimal(18,2))") <= 0, None)
    .otherwise(expr("try_cast(receita_usd as decimal(18,2))"))
)

# Cotação do Dólar

Construção da série temporal contínua de cotações com forward-fill para preencher dias sem dados.

In [0]:
from pyspark.sql.functions import col, to_date, last, min, max, lit
from pyspark.sql.window import Window

# Lê a tabela crua da camada Bronze
df_cotacao_bronze = spark.read.table("bronze.tb_cotacao_dolar")

# Converte a string de dataHoraCotacao para Date e tipa a cotação
df_cotacao_base = df_cotacao_bronze.withColumn(
    "data_cotacao", to_date(col("dataHoraCotacao"))
).withColumn(
    "cotacao_dolar_original", col("cotacaoCompra").cast("decimal(10,4)")
).select("data_cotacao", "cotacao_dolar_original").distinct()

# Encontra a data mínima e máxima disponível na Bronze
limites = df_cotacao_base.agg(
    min("data_cotacao").alias("min_data"), 
    max("data_cotacao").alias("max_data")
).collect()[0]

# Cria um DataFrame com uma sequência contínua de dias (série temporal)
df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{limites['min_data']}'), to_date('{limites['max_data']}'), interval 1 day)) as data_referencia
""")

# Join e Forward Fill: preenche os dias sem cotação com a última cotação válida
janela_ffill = Window.partitionBy(lit(1)).orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao = df_calendario.join(
    df_cotacao_base,
    df_calendario.data_referencia == df_cotacao_base.data_cotacao,
    "left"
).withColumn(
    "cotacao_dolar",
    last("cotacao_dolar_original", ignorenulls=True).over(janela_ffill)
).select("data_referencia", "cotacao_dolar")

# Salva na camada Silver
(
    df_silver_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_cotacao_dolar")
)

In [0]:
# Cruzamento com Info Filmes e Cotação
df_info_filmes = spark.read.table("silver.tb_info_filmes")
df_cotacao_dolar = spark.read.table("silver.tb_cotacao_dolar")

df_fin_cruzado = df_fin_decimal.join(
    df_info_filmes.select("id_filme", "data_lancamento"),
    on="id_filme",
    how="left"
).join(
    df_cotacao_dolar,
    df_info_filmes.data_lancamento == df_cotacao_dolar.data_referencia,
    how="left"
)

In [0]:
# conversão de moeda
df_silver_financeiro = df_fin_cruzado.withColumn(
    "orcamento_brl", (col("orcamento_usd") * col("cotacao_dolar")).cast("decimal(18,2)")
).withColumn(
    "receita_brl", (col("receita_usd") * col("cotacao_dolar")).cast("decimal(18,2)")
).withColumn(
    "lucro_usd", col("receita_usd") - col("orcamento_usd")
).withColumn(
    "lucro_brl", col("receita_brl") - col("orcamento_brl")
).withColumn(
    "margem_lucro_percentual",
    when(col("receita_usd") > 0, (col("lucro_usd") / col("receita_usd")) * 100)
    .otherwise(None).cast("decimal(10,2)")
).select(
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", 
    "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_percentual"
)

In [0]:
# Salvar Tabela Silver
(
    df_silver_financeiro.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_financeiro_filmes")
)

# Processamento: tb_metricas_engajamento

Deduplicação, limpeza textual e try-cast blindado contra column shift para métricas de engajamento.

In [0]:
from pyspark.sql.functions import col, row_number, when, regexp_replace, expr
from pyspark.sql.window import Window

# Leitura da Bronze e Deduplicação pelo id
df_bronze_metrics = spark.read.table("bronze.tb_movies_metrics")
janela_metrics = Window.partitionBy("id").orderBy(col("ingestion_datetime").desc())

df_metrics_dedup = (
    df_bronze_metrics
    .withColumn("row_num", row_number().over(janela_metrics))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
# Traducao e Renomeacao de Colunas - Metricas
df_metrics_traduzido = (
    df_metrics_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
    .withColumnRenamed("popularity", "popularidade")
)

In [0]:
# Limpeza Textual Inicial
df_metrics_limpo = (
    df_metrics_traduzido
    .withColumn("nota_media_tmdb", regexp_replace(regexp_replace(col("nota_media_tmdb"), ",", "."), "[^0-9.]", ""))
    .withColumn("qtd_votos_tmdb", regexp_replace(col("qtd_votos_tmdb"), "[^0-9]", ""))
    .withColumn("nota_media_imdb", regexp_replace(regexp_replace(col("nota_media_imdb"), ",", "."), "[^0-9.]", ""))
    .withColumn("qtd_votos_imdb", regexp_replace(col("qtd_votos_imdb"), "[^0-9]", ""))
    .withColumn("popularidade", regexp_replace(col("popularidade"), "[^0-9.]", ""))
)

In [0]:
from pyspark.sql.functions import expr, when

# Aplicacao do Try-Cast para Blindagem contra Column Shift
df_silver_metrics_treated = (
    df_metrics_limpo
    .withColumn(
        "nota_media_tmdb",
        when((expr("try_cast(nota_media_tmdb as double)") >= 0) & (expr("try_cast(nota_media_tmdb as double)") <= 10), expr("try_cast(nota_media_tmdb as double)"))
        .otherwise(None)
    )
    .withColumn(
        "qtd_votos_tmdb",
        when(expr("try_cast(qtd_votos_tmdb as int)") >= 0, expr("try_cast(qtd_votos_tmdb as int)"))
        .otherwise(None)
    )
    .withColumn(
        "nota_media_imdb",
        when((expr("try_cast(nota_media_imdb as double)") >= 0) & (expr("try_cast(nota_media_imdb as double)") <= 10), expr("try_cast(nota_media_imdb as double)"))
        .otherwise(None)
    )
    .withColumn(
        "qtd_votos_imdb",
        when(expr("try_cast(qtd_votos_imdb as int)") >= 0, expr("try_cast(qtd_votos_imdb as int)"))
        .otherwise(None)
    )
    .withColumn(
        "popularidade",
        when(expr("try_cast(popularidade as double)") >= 0, expr("try_cast(popularidade as double)"))
        .otherwise(None)
    )
)

In [0]:
# Persistência na Camada Silver
(
    df_silver_metrics_treated.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_metricas_engajamento")
)



# Processamento: tb_avaliacoes_usuarios

Renomeação, deduplicação e aplicação de regras de negócio (escala 0–10 e padronização de comentários).

In [0]:
#Renomear Tabela Avaliações de usuários e deduplicar
df_reviews_bronze = spark.read.table("bronze.tb_movies_reviews")
df_reviews_silver = (
    df_reviews_bronze
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]
)
)


In [0]:
from pyspark.sql.functions import col, when, trim

'''Aplicação da regra de negocio da escala permitida (0 a 10).
Qualquer valor fora dessa faixa deve ser descartado e convertido para NULL. '''

df_avaliacoes_tratadas = df_reviews_silver.withColumn(
    "nota_usuario",
    when((col("nota_usuario") >= 0) & (col("nota_usuario") <= 10), col("nota_usuario"))
    .otherwise(None)
).withColumn(
    "comentario_usuario",
    when(trim(col("comentario_usuario")) == "", "Sem comentário")
    .when(col("comentario_usuario").isNull(), "Sem comentário")
    .otherwise(trim(col("comentario_usuario")))
)

In [0]:
# Guardar na Silver
(
    df_avaliacoes_tratadas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_avaliacoes_usuarios")
)

# Processamento: tb_generos e tb_pessoas_empresas

Explosão e limpeza de gêneros, e extração de entidades (atores, diretores, roteiristas e produtoras).

In [0]:
from pyspark.sql.functions import col, regexp_replace, split, explode, trim, expr

df_credits_bronze = spark.read.table("bronze.tb_credits_and_tags")

# Normalizar separador, separar por vírgula e explodir para linhas únicas
df_generos_explodidos = df_credits_bronze.select(
    col("id").alias("id_filme"),
    explode(split(regexp_replace(col("genres"), ";", ","), ",")).alias("nome_genero")
)

# Limpeza e remoção de ruídos (Column shift)
df_generos_tratados = df_generos_explodidos.withColumn(
    "nome_genero", trim(col("nome_genero"))
).filter(
    (col("nome_genero") != "") & 
    col("nome_genero").isNotNull() & 
    expr("try_cast(nome_genero as double)").isNull() # Remove números deslocados
).dropDuplicates(["id_filme", "nome_genero"])

# Guardar na Silver
(
    df_generos_tratados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_generos")
)

In [0]:
from pyspark.sql.functions import col, lit, initcap, trim, explode, split, regexp_replace

# Função auxiliar para extrair e identificar o tipo da entidade
def extrair_entidades(df, coluna_origem, tipo_entidade):
    return df.select(
        col("id").alias("id_filme"),
        explode(split(regexp_replace(col(coluna_origem), ";", ","), ",")).alias("nome_entidade")
    ).withColumn("tipo_entidade", lit(tipo_entidade))

# Extração de cada tipo
df_atores = extrair_entidades(df_credits_bronze, "cast", "Ator")
df_diretores = extrair_entidades(df_credits_bronze, "directors", "Diretor")
df_roteiristas = extrair_entidades(df_credits_bronze, "writers", "Roteirista")
df_produtoras = extrair_entidades(df_credits_bronze, "production_companies", "Produtora")

# União e limpeza (Capitalização e remoção de nulos/vazios)
df_entidades_unificadas = (
    df_atores.union(df_diretores)
    .union(df_roteiristas)
    .union(df_produtoras)
)

df_pessoas_empresas = df_entidades_unificadas.withColumn(
    "nome_entidade", initcap(trim(col("nome_entidade")))
).filter(
    (col("nome_entidade") != "") & col("nome_entidade").isNotNull()
).dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])

# Guardar na Silver
(
    df_pessoas_empresas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_pessoas_empresas")
)

In [0]:
spark.sql("USE CATALOG cinedata_lakehouse")
spark.sql("USE SCHEMA silver")

resumo = spark.sql("""
    SELECT 'tb_info_filmes' as tabela, COUNT(*) as linhas FROM silver.tb_info_filmes
    UNION ALL SELECT 'tb_financeiro_filmes', COUNT(*) FROM silver.tb_financeiro_filmes
    UNION ALL SELECT 'tb_cotacao_dolar', COUNT(*) FROM silver.tb_cotacao_dolar
    UNION ALL SELECT 'tb_metricas_engajamento', COUNT(*) FROM silver.tb_metricas_engajamento
    UNION ALL SELECT 'tb_avaliacoes_usuarios', COUNT(*) FROM silver.tb_avaliacoes_usuarios
    UNION ALL SELECT 'tb_generos', COUNT(*) FROM silver.tb_generos
    UNION ALL SELECT 'tb_pessoas_empresas', COUNT(*) FROM silver.tb_pessoas_empresas
""")
display(resumo)